# Tutorial: write `src/preprocess.py` (no pandas)

`eda.ipynb` **decided** the rules. This notebook **implements** them, one function per block.

You will copy the finished functions into the blank file `ML_project_1/src/preprocess.py`.

Pipeline (same for train and test, **statistics only from train**):

```text
raw x  →  drop bad columns  →  sentinels → nan/0  →  median impute  →  standardize
```

Allowed: `os`, `numpy`. Forbidden: pandas, sklearn.

Kernel: **`ml`**. First real load uses `data/cache_eda.npz` from EDA (fast).
This is the **only** preprocess tutorial. A second draft (`doc/preprocess.ipynb`) was the same lesson and was removed.


## 0. Paths and a tiny toy matrix

We practise each idea on 4 rows × 4 columns **before** the 328k table.


In [1]:
import sys
from pathlib import Path

import numpy as np

DOC = Path(".").resolve()
ROOT = DOC.parent if DOC.name == "doc" else DOC
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))

print("ROOT", ROOT)


ROOT /Users/yijunliu/Documents/ML/ML_project_1


In [ ]:
# Toy: 4 people, 4 features. nan = empty cell.
# columns: admin_id | age_group | sick_days | leak_rehab
toy_names = np.array(["SEQNO", "_AGEG5YR", "PHYSHLTH", "HAREHAB1"])
toy = np.array(
    [
        [10.0,  9.0, 88.0, np.nan],  # 88 on PHYSHLTH = "zero days"
        [11.0, 14.0,  5.0,    1.0],  # age 14 = missing; rehab answered
        [12.0,  3.0, 99.0, np.nan],  # 99 = refused
        [13.0,  7.0,  2.0, np.nan],  # age 7 is a REAL age group, keep it
    ]
)
print(toy)
print("names", toy_names)


## 1. Read the 321 feature names

`x` has **no** header. Names are row 0 of `x_train.csv`, skipping `Id`.


In [ ]:
def load_feature_names(data_path):
    """Return 1D array of 321 column names (no Id)."""
    path = Path(data_path) / "x_train.csv"
    with open(path) as f:
        header = f.readline().strip().split(",")
    return np.array(header[1:])


names = load_feature_names(DATA)
print(len(names), names[:8], "...", names[-3:])


## 2. Drop columns by **name**

`np.isin(names, DROP_COLS)` is a boolean mask of length 321.

- `keep = ~np.isin(names, drop_cols)`
- `x[:, keep]` keeps those columns for **every row**
- Apply the **same** `keep` to `x_test`

`DROP_COLS` is the EDA list: admin, constants, leakage, duplicates, >90% nan.


In [ ]:
DROP_COLS = [
    "_STATE", "FMONTH", "IDATE", "IMONTH", "IDAY", "IYEAR", "DISPCODE", "SEQNO", "_PSU",
    "CTELENUM", "PVTRESD1", "COLGHOUS", "STATERES", "CELLFON3", "LADULT", "NUMADULT",
    "NUMMEN", "NUMWOMEN", "CTELNUM1", "CELLFON2", "CADULT", "PVTRESD2", "CCLGHOUS",
    "CSTATE", "LANDLINE", "HHADULT", "QSTVER", "QSTLANG", "MSCODE", "_STSTR", "_STRWT",
    "_RAWRAKE", "_WT2RAKE", "_CHISPNC", "_CRACE1", "_CPRACE", "_CLLCPWT", "_DUALUSE",
    "_DUALCOR", "_LLCPWT",
    "HAREHAB1", "STREHAB1", "CVDASPRN", "ASPUNSAF", "RLIVPAIN", "RDUCHART",
    "WEIGHT2", "HEIGHT3", "HTIN4", "HTM4", "WTKG3", "_BMI5CAT", "_RFBMI5",
    "NUMPHON2", "INSULIN", "BLDSUGAR", "FEETCHK2", "DOCTDIAB", "CHKHEMO3", "FEETCHK",
    "EYEEXAM", "DIABEYE", "DIABEDU", "CRGVREL1", "CRGVLNG1", "CRGVHRS1", "CRGVPRB1",
    "CRGVPERS", "CRGVHOUS", "CRGVMST2", "VIDFCLT2", "VIREDIF3", "VIPRFVS2", "VINOCRE2",
    "VIEYEXM2", "VIINSUR2", "VICTRCT4", "VIGLUMA2", "VIMACDG2", "CDHOUSE", "CDASSIST",
    "CDHELP", "CDSOCIAL", "CDDISCUS", "WTCHSALT", "LONGWTCH", "DRADVISE", "ASTHMAGE",
    "ASATTACK", "ASERVIST", "ASDRVIST", "ASRCHKUP", "ASACTLIM", "ASYMPTOM", "ASNOSLEP",
    "ASTHMED3", "ASINHALR", "RDUCSTRK", "ARTTODAY", "ARTHWGT", "ARTHEXER", "ARTHEDU",
    "TETANUS", "HPVADVC2", "HPVADSHT", "SHINGLE2", "HADMAM", "HOWLONG", "HADPAP2",
    "LASTPAP2", "HPVTEST", "HPLSTTST", "HADHYST2", "PROFEXAM", "LENGEXAM", "LSTBLDS3",
    "HADSGCO1", "LASTSIG3", "PCPSAAD2", "PCPSADI1", "PCPSARE1", "PSATEST1", "PSATIME",
    "PCPSARS1", "PCPSADE1", "PCDMDECN", "SCNTPAID", "SCNTWRK1", "SCNTLPAD", "SCNTLWK1",
    "CASTHNO2", "EMTSUPRT", "LSATISFY", "ADPLEASR", "ADDOWN", "ADSLEEP", "ADENERGY",
    "ADEAT1", "ADFAIL", "ADTHINK", "ADMOVE", "MISTMNT", "ADANXEV",
]


def drop_features(x, names, drop_cols=None):
    """Return (x_kept, names_kept). Same columns for any x with this `names`."""
    if drop_cols is None:
        drop_cols = DROP_COLS
    drop_cols = np.asarray(drop_cols)
    keep = ~np.isin(names, drop_cols)
    return x[:, keep], names[keep]


toy_kept, toy_kept_names = drop_features(toy, toy_names, drop_cols=["SEQNO", "HAREHAB1"])
print("kept names", toy_kept_names)
print(toy_kept)


On the toy we dropped admin `SEQNO` and leakage `HAREHAB1`. Age and sick-days stay.

`x[:, keep]` is **boolean indexing** on columns: `keep` has length = number of columns.


## 3. Sentinels (the careful part)

Work on a **copy** (`x = np.array(x, dtype=float, copy=True)`). Loop columns by name.

| Column | Rule |
|---|---|
| `PHYSHLTH`, `MENTHLTH`, `POORHLTH`, `CHILDREN` | `88` → **0** (none); `77`,`99` → `nan` |
| `_AGEG5YR` | only `14` → `nan`. **Keep 7 and 9** (real ages) |
| `_AGE65YR`, `_AGE80`, `_AGE_G`, `_BMI5` | do **not** treat 7/9 as missing |
| other kept columns | `77,99,777,999,7777,9999` → `nan`; also `7,9` → `nan` (DK/refused) |


In [ ]:
SENTINEL_DEFAULT = (77, 99, 777, 999, 7777, 9999)
SENTINEL_DK_REFUSED = (7, 9)
DAYS_NONE_TO_ZERO = ("PHYSHLTH", "MENTHLTH", "POORHLTH", "CHILDREN")
NEVER_TREAT_7_9_AS_MISSING = ("_AGEG5YR", "_AGE65YR", "_AGE80", "_AGE_G", "_BMI5")


def apply_sentinels(x, names):
    """Replace BRFSS codes. Returns a new array; does not drop columns."""
    x = np.array(x, dtype=float, copy=True)
    names = np.asarray(names)
    for j, name in enumerate(names):
        col = x[:, j]
        if name in DAYS_NONE_TO_ZERO:
            col = np.where(col == 88, 0.0, col)
            col = np.where(np.isin(col, [77, 99]), np.nan, col)
        elif name == "_AGEG5YR":
            col = np.where(col == 14, np.nan, col)
        else:
            col = np.where(np.isin(col, SENTINEL_DEFAULT), np.nan, col)
            if name not in NEVER_TREAT_7_9_AS_MISSING:
                col = np.where(np.isin(col, SENTINEL_DK_REFUSED), np.nan, col)
        x[:, j] = col
    return x


toy_s = apply_sentinels(toy_kept, toy_kept_names)
print("names", toy_kept_names)
print("after sentinels\n", toy_s)


Check the toy:

- row0 `PHYSHLTH` 88 → **0**
- row1 age **14** → `nan`
- row2 `PHYSHLTH` 99 → `nan`
- row3 age **7** still **7**


## 4. Median impute — `nanmedian` along rows

For each column `j`, the fill value is the median of **train** values that are not `nan`.

```text
medians.shape == (D,)
x[:, j]  ←  medians[j]   wherever x[:, j] is nan
```

`np.where(np.isnan(x), medians, x)` works because `medians` broadcasts from `(D,)` onto `(N, D)`.


In [ ]:
def fit_medians(x):
    """1D medians, one per column. Ignores nan."""
    return np.nanmedian(x, axis=0)


def apply_medians(x, medians):
    x = np.array(x, dtype=float, copy=True)
    return np.where(np.isnan(x), medians, x)


med = fit_medians(toy_s)
print("medians", med)
print("imputed\n", apply_medians(toy_s, med))


If a whole column is `nan` (should not happen after our drops), `nanmedian` is `nan`. Guard with `np.nan_to_num(medians, nan=0.0)` inside `fit_medians` for safety.


In [ ]:
def fit_medians(x):
    med = np.nanmedian(x, axis=0)
    return np.nan_to_num(med, nan=0.0)


## 5. Standardize — mean and std of **imputed train**

$$
x_{ij} \leftarrow \frac{x_{ij} - \mu_j}{\sigma_j}
$$

- `mu`, `sigma` from **train after impute**
- if `sigma_j == 0`, use `1` so we do not divide by 0 (column becomes all zeros after subtracting the mean)
- **test** uses the **same** `mu` and `sigma` (never recompute on test)


In [ ]:
def fit_standardize(x_imputed):
    """Return (mean, std) per column. std==0 is replaced by 1."""
    mean = np.mean(x_imputed, axis=0)
    std = np.std(x_imputed, axis=0)
    std = np.where(std == 0, 1.0, std)
    return mean, std


def apply_standardize(x_imputed, mean, std):
    return (x_imputed - mean) / std


imp = apply_medians(toy_s, fit_medians(toy_s))
mean, std = fit_standardize(imp)
z = apply_standardize(imp, mean, std)
print("mean~0", np.round(z.mean(axis=0), 6))
print("std~1 ", np.round(z.std(axis=0), 6))
print(z)


## 6. One `fit` / `transform` so test cannot leak

**Wrong:** compute medians on train and test stacked together.  
**Right:** `fit` on train only, `transform` test.

`fit` returns everything `transform` needs: kept names, medians, mean, std.


In [ ]:
def fit_preprocess(x_train, names, drop_cols=None):
    """Learn drop mask + impute + scale on TRAIN only.

    Returns dict: names_kept, medians, mean, std
    and the transformed x_train.
    """
    x, names_kept = drop_features(x_train, names, drop_cols)
    x = apply_sentinels(x, names_kept)
    medians = fit_medians(x)
    x = apply_medians(x, medians)
    mean, std = fit_standardize(x)
    x = apply_standardize(x, mean, std)
    stats = {
        "names_kept": names_kept,
        "medians": medians,
        "mean": mean,
        "std": std,
        "drop_cols": DROP_COLS if drop_cols is None else drop_cols,
    }
    return x, stats


def transform_preprocess(x, names, stats):
    """Apply a fitted pipeline to val or test."""
    x, names_kept = drop_features(x, names, stats["drop_cols"])
    if not np.array_equal(names_kept, stats["names_kept"]):
        raise ValueError("column names after drop do not match the train fit.")
    x = apply_sentinels(x, names_kept)
    x = apply_medians(x, stats["medians"])
    x = apply_standardize(x, stats["mean"], stats["std"])
    return x


In [ ]:
# Toy: pretend first 3 rows = train, last row = test
x_tr_toy, stats_toy = fit_preprocess(toy[:3], toy_names, drop_cols=["SEQNO", "HAREHAB1"])
x_te_toy = transform_preprocess(toy[3:], toy_names, stats_toy)
print("train z\n", x_tr_toy)
print("test z\n", x_te_toy)
print("kept", stats_toy["names_kept"])


## 6b. Split train / val **before** `fit_preprocess`

If you compute medians on all 328k rows, then hold out validation, those val rows already leaked into the median.

Do this in `run.py` (you may also copy `train_val_split` into `preprocess.py`):

1. shuffle indices
2. split `x`, `y`, `ids`
3. `fit_preprocess` on the **train slice only**
4. `transform_preprocess` on val and test


In [ ]:
def train_val_split(x, y, ids, val_ratio=0.2, seed=0):
    """Shuffle then split. Returns x_tr, y_tr, ids_tr, x_val, y_val, ids_val."""
    n = x.shape[0]
    rng = np.random.RandomState(seed)
    perm = rng.permutation(n)
    n_val = int(n * val_ratio)
    val_i, tr_i = perm[:n_val], perm[n_val:]
    return x[tr_i], y[tr_i], ids[tr_i], x[val_i], y[val_i], ids[val_i]


# Toy: 4 rows already in `toy`; split 1 val row
y_toy = np.array([-1, 1, -1, -1])
ids_toy = np.array([0, 1, 2, 3])
x_tr, y_tr, id_tr, x_va, y_va, id_va = train_val_split(
    toy, y_toy, ids_toy, val_ratio=0.25, seed=0
)
print("train rows", x_tr.shape[0], "val rows", x_va.shape[0])


## 7. Run on the real cache (optional check)

Should end with train shape `(328135, 178)` and **no nan**.


In [ ]:
cache = DATA / "cache_eda.npz"
if cache.exists():
    z = np.load(cache)
    x_raw, x_te_raw = z["x"], z["x_te"]
    x_tr, stats = fit_preprocess(x_raw, names)
    x_te = transform_preprocess(x_te_raw, names, stats)
    print("train", x_tr.shape, "test", x_te.shape)
    print("nan train", np.isnan(x_tr).sum(), "nan test", np.isnan(x_te).sum())
    print("train mean abs (should be ~0)", np.mean(np.abs(x_tr.mean(axis=0))))
else:
    print("no cache; skip. Run eda.ipynb once or load_csv_data.")


## 8. Optional bias column

`least_squares` / GD expect `tx` to include a **column of ones** if you want an intercept.

Do this in `run.py` **after** preprocess, or add a tiny helper:


In [ ]:
def add_bias(x):
    """Prepend a column of 1s. Shape (N, D) -> (N, D+1)."""
    ones = np.ones((x.shape[0], 1))
    return np.hstack([ones, x])


print(add_bias(np.array([[2.0, 3.0], [4.0, 5.0]])))


## 9. Copy into `src/preprocess.py`

Create `/ML_project_1/src/preprocess.py` with:

1. `import numpy as np` (and `from pathlib import Path` if you keep `load_feature_names`)
2. the constants: `DROP_COLS`, `SENTINEL_*`, `DAYS_NONE_TO_ZERO`, `NEVER_TREAT_7_9_AS_MISSING`
3. functions: `load_feature_names`, `drop_features`, `apply_sentinels`, `fit_medians`, `apply_medians`, `fit_standardize`, `apply_standardize`, `fit_preprocess`, `transform_preprocess`, `add_bias`, `train_val_split`

Typical use later in `run.py`:

```python
from pathlib import Path
from helpers import load_csv_data
from src.preprocess import load_feature_names, fit_preprocess, transform_preprocess, add_bias

y, x, ids_tr, x_te, ids_te = load_csv_data("data")
names = load_feature_names("data")
x, stats = fit_preprocess(x, names)
x_te = transform_preprocess(x_te, names, stats)
tx = add_bias(x)
```

If `from src.preprocess import ...` fails, either run from the repo root, or put `preprocess.py` next to `helpers.py` and `from preprocess import ...`.

Do **not** split train/val inside preprocess (medians would be wrong if you split after). Split **indices first**, then `fit_preprocess(x[train])` and `transform_preprocess(x[val], stats)`.

Labels `y` are not modified here (`+1/-1`). Map to `{0,1}` only when you call `logistic_regression`.
